In [ ]:
#| default_exp a2wav

In [ ]:
#| exporti
from vocos import Vocos
from whisperspeech import inference
import numpy as np
import soundfile as sf
import torch

In [ ]:
#| export
def save_audio(fname, audio, sample_rate=24000, normalize=True, peak=0.9):
    "Save `audio` (a `(channels, samples)` tensor or array) to `fname`, format picked from the extension."
    # soundfile instead of torchaudio.save: recent torchaudio routes saving through
    # torchcodec, which needs a matching FFmpeg shared library and fails on a plain install.
    audio = audio.detach().cpu().float().numpy() if torch.is_tensor(audio) else np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2: audio = audio.T  # soundfile expects (samples, channels)
    # The model output level varies a lot between languages and voices (English with the
    # default speaker comes out ~6x quieter than Polish and sounds like a hiss), so we
    # peak-normalize by default. Silence is left as is to avoid dividing by zero.
    if normalize:
        m = np.abs(audio).max()
        if m > 0: audio = audio * (peak / m)
    sf.write(fname, audio, sample_rate)

In [ ]:
#| export
class Vocoder:
    def __init__(self, repo_id="charactr/vocos-encodec-24khz", device=None, cache_dir=None):
        if device is None: device = inference.get_compute_device()
        if device == 'mps': device = 'cpu' # mps does not currently work with vocos, thus only cuda or cpu
        self.device = device
        self.vocos = Vocos.from_pretrained(repo_id).to(device)

    def is_notebook(self):
        try:
            return get_ipython().__class__.__name__ == "ZMQInteractiveShell"
        except:
            return False

    @torch.no_grad()
    def decode(self, atoks):
        if len(atoks.shape) == 3:
            b,q,t = atoks.shape
            
            atoks = atoks.permute(1,0,2)
        else:
            q,t = atoks.shape
        # on mps we run Vocos on the CPU, make sure it's input is on the correct device
        atoks = atoks.to(self.device)
        # print(atoks.dtype, atoks.device) # uncomment to check dtype and compute_device
        features = self.vocos.codes_to_features(atoks)
        bandwidth_id = torch.tensor({2: 0, 4: 1, 8: 2}[q]).to(self.device)  # Move tensor to the same device as model
        return self.vocos.decode(features, bandwidth_id=bandwidth_id)
        
    def decode_to_file(self, fname, atoks, normalize=True):
        save_audio(fname, self.decode(atoks), 24000, normalize=normalize)
        if self.is_notebook():
            from IPython.display import display, HTML, Audio
            display(HTML(f'<a href="{fname}" target="_blank">Listen to {fname}</a>'))
        
    def decode_to_notebook(self, atoks):
        from IPython.display import display, HTML, Audio

        audio = self.decode(atoks)
        display(Audio(audio.cpu().numpy(), rate=24000))

In [ ]:
import os, tempfile
# Regression tests for save_audio. Each case is isolated and an exception counts as a failed
# check rather than a crash, so a regression is distinguishable from a broken environment.
t = torch.linspace(0, 1, 24000)
quiet = (0.05*torch.sin(2*torch.pi*220*t)).unsqueeze(0)  # (channels, samples), like Vocos output

def _peak_after_save(fname, audio, **kwargs):
    save_audio(fname, audio, **kwargs)
    a, sr = sf.read(fname)
    assert sr == 24000, f'sample rate {sr}'
    return np.abs(a).max()

failures = []
with tempfile.TemporaryDirectory() as d:
    # all formats must be writable without FFmpeg/torchcodec, and quiet input must be peak-normalized
    cases = [(f'{ext} saved and normalized', lambda ext=ext: abs(_peak_after_save(os.path.join(d, f't.{ext}'), quiet) - 0.9) < 0.1)
             for ext in ['wav', 'flac', 'ogg', 'mp3']]
    cases += [('normalize=False keeps the level', lambda: abs(_peak_after_save(os.path.join(d, 'raw.wav'), quiet, normalize=False) - 0.05) < 0.01),
              ('silence stays silent', lambda: _peak_after_save(os.path.join(d, 'zero.wav'), torch.zeros(1, 2400)) == 0)]
    for name, check in cases:
        try: ok = check()
        except Exception as e: ok, name = False, f'{name} ({type(e).__name__}: {str(e).splitlines()[0][:80]})'
        if not ok: failures.append(name)
assert not failures, 'save_audio failures: ' + '; '.join(failures)
print(f'save_audio: {len(cases)} checks passed')